In [25]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

import helpers
import preprocessing

In [ ]:
data_path = '../data/dataset'
x_train, x_test, y_train, train_ids, test_ids = helpers.load_csv_data(data_path)



In [ ]:
display(x_train)
display(y_train)

df = pd.DataFrame(x_train)
df.head()

array([[5.3000000e+01, 1.1000000e+01, 1.1162015e+07, ...,           nan,
                  nan, 2.0000000e+00],
       [3.3000000e+01, 1.2000000e+01, 1.2152015e+07, ...,           nan,
                  nan,           nan],
       [2.0000000e+01, 1.0000000e+01, 1.0202015e+07, ..., 1.0000000e+00,
        2.0000000e+00, 2.0000000e+00],
       ...,
       [3.9000000e+01, 1.0000000e+01, 1.0202015e+07, ..., 2.0000000e+00,
        2.0000000e+00, 2.0000000e+00],
       [3.3000000e+01, 1.2000000e+01, 1.2302015e+07, ...,           nan,
                  nan, 2.0000000e+00],
       [3.2000000e+01, 9.0000000e+00, 9.1220150e+06, ...,           nan,
                  nan, 2.0000000e+00]], shape=(328135, 321))

array([-1, -1, -1, ..., -1, -1, -1], shape=(328135,))

,0,1,2,3,4,5,6,7,8,9,...,311,312,313,314,315,316,317,318,319,320
0,53.0,11.0,11162015.0,11.0,16.0,2015.0,1100.0,2.015016e+09,2.015016e+09,NaN,...,1.0,1.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0
1,33.0,12.0,12152015.0,12.0,15.0,2015.0,1200.0,2.015004e+09,2.015004e+09,1.0,...,9.0,9.0,3.0,3.0,4.0,9.0,9.0,NaN,NaN,NaN
2,20.0,10.0,10202015.0,10.0,20.0,2015.0,1100.0,2.015006e+09,2.015006e+09,1.0,...,4.0,2.0,2.0,2.0,3.0,1.0,1.0,1.0,2.0,2.0
3,42.0,6.0,6182015.0,6.0,18.0,2015.0,1100.0,2.015005e+09,2.015005e+09,NaN,...,2.0,2.0,2.0,2.0,3.0,1.0,1.0,2.0,2.0,2.0
4,24.0,11.0,11062015.0,11.0,6.0,2015.0,1100.0,2.015005e+09,2.015005e+09,1.0,...,9.0,9.0,3.0,3.0,4.0,1.0,1.0,9.0,9.0,2.0


## Preprocessing

Goal: turn the raw BRFSS 2015 table (321 columns, full of answer codes such as 7 = "don't know") into a clean numeric matrix for the models.

- `preprocessing.py` is the version the models use: numpy only, checked by `python test_preprocessing.py`.
- This notebook walks through the same steps on `x_train` (with pandas, to see what each step does), then step 6 checks that both give the same features.

`preprocessing.py` works in two phases:
- **fit** (`fit_preprocessing`) on the training rows: learn every statistic (codes, kept columns, medians, clip bounds, categories, mean, std)
- **transform** (`transform`) any rows (train, validation, test) with those statistics only, so nothing leaks from the rows we predict

In cross-validation, fit on each training fold, not on the whole training set.

Meaning of every column and answer code: `data/dataset/codebook15_llcp.pdf` (BRFSS 2015 codebook, search the column name).

Expected result on the full training set: 157 of the 321 columns kept, 348 features, no NaN. On an 80/20 holdout with ridge (λ = 1e-3): F1 0.422 / AUC 0.855, against 0.412 / 0.845 with plain mean imputation.

### Step 0: column names

`load_csv_data` returns the arrays without the header, but most steps below work by column name.

- Read the first line of `data/dataset/x_train.csv`, split it on commas, drop the first name (`Id`). `preprocessing.load_feature_names` does the same.
- That gives 321 names, in the same order as the columns of `x_train`.
- Check that `x_train.shape[1] == len(col_names)`.

**Reading the names.** They are CDC's own variable names (the "SAS Variable Name" of each entry in `data/dataset/codebook15_llcp.pdf`); the course staff only added `Id`. So every column can be looked up by name in the codebook.
- **At most 8 characters** (an old SAS limit), hence the abbreviations: `CVDINFR4` = cardiovascular disease, infarction (heart attack).
- **Trailing digit = version of the question**, raised when the wording changes: `ALCDAY5` is the 5th version of the alcohol question, `WEIGHT2` the 2nd version of the weight question.
- **Underscore at the start or the end = calculated by CDC**, not asked directly: `_MICHD` (our target), `_RFHYPE5` (high blood pressure yes/no), `DROCDY3_` (drinks per day, decoded from `ALCDAY5`). Names without an underscore are the raw answers.

When a raw answer and an underscore column cover the same thing (e.g. `BPHIGH4` and `_RFHYPE5`), the underscore one is usually already cleaned, and the two are near-copies of each other.

The codebook's "Column: 23-26" field is the position of the variable in CDC's raw fixed-width text file (characters 23 to 26 of each line), not several columns: ignore it, every variable is one column in our CSV.

In [ ]:
# Header of x_train.csv without 'Id': same order as the columns of x_train
with open(f"{data_path}/x_train.csv") as f:
    col_names = f.readline().strip().split(",")[1:]
assert x_train.shape[1] == len(col_names)
df = pd.DataFrame(x_train, columns=col_names)

### Step 1: drop columns that say nothing about the respondent's health

Drop these by name.

**Survey bookkeeping** (dates, ids, phone screening, sampling weights) and questions about a child of the household:

`FMONTH IDATE IMONTH IDAY IYEAR DISPCODE SEQNO _PSU CTELENUM PVTRESD1 COLGHOUS STATERES CELLFON3 LADULT CTELNUM1 CELLFON2 CADULT PVTRESD2 CCLGHOUS CSTATE LANDLINE NUMHHOL2 NUMPHON2 CPDEMO1 QSTVER QSTLANG _STSTR _STRWT _RAWRAKE _WT2RAKE _DUALUSE _DUALCOR _LLCPWT _CLLCPWT _CHISPNC _CRACE1 _CPRACE`

**Raw answers that pack a unit and an amount into one number**: the first digit is the unit, the other digits are the amount. In `ALCDAY5`, 105 = 5 days per week and 215 = 15 days in the past 30 days, so the larger number is actually less drinking. In `WEIGHT2`, 180 = 180 pounds but 9080 = 80 kg. The same group also holds two dates stored as one number (`FLSHTMY2`, `HIVTSTD3`: 072014 = July 2014) and the type of exercise (`EXRACT11`, `EXRACT21`: about 75 unordered labels such as 5 = basketball). BRFSS already provides decoded versions of most of them, in one unit (`WTKG3`, `HTM4`, `DROCDY3_`, `FRUTDA1_`, `PAFREQ1_`, `STRFREQ_`, ...):

`WEIGHT2 HEIGHT3 ALCDAY5 FRUITJU1 FRUIT1 FVBEANS FVGREEN FVORANG VEGETAB1 EXRACT11 EXEROFT1 EXERHMM1 EXRACT21 EXEROFT2 EXERHMM2 STRENGTH FLSHTMY2 HIVTSTD3 BLDSUGAR FEETCHK2 LONGWTCH`

In [ ]:
# Survey bookkeeping (dates, ids, phone screening, sampling weights) and
# questions about a child of the household rather than the respondent
ADMIN_COLUMNS = set(
    """
    FMONTH IDATE IMONTH IDAY IYEAR DISPCODE SEQNO _PSU CTELENUM PVTRESD1
    COLGHOUS STATERES CELLFON3 LADULT CTELNUM1 CELLFON2 CADULT PVTRESD2
    CCLGHOUS CSTATE LANDLINE NUMHHOL2 NUMPHON2 CPDEMO1 QSTVER QSTLANG _STSTR
    _STRWT _RAWRAKE _WT2RAKE _DUALUSE _DUALCOR _LLCPWT _CLLCPWT
    _CHISPNC _CRACE1 _CPRACE
    """.split()
)

# Raw answers that pack a unit and an amount (or a date) into one number.
# BRFSS provides decoded versions of most of them (WTKG3, HTM4, DROCDY3_, ...)
SUPERSEDED_COLUMNS = set(
    """
    WEIGHT2 HEIGHT3 ALCDAY5 FRUITJU1 FRUIT1 FVBEANS FVGREEN FVORANG VEGETAB1
    EXRACT11 EXEROFT1 EXERHMM1 EXRACT21 EXEROFT2 EXERHMM2 STRENGTH
    FLSHTMY2 HIVTSTD3 BLDSUGAR FEETCHK2 LONGWTCH
    """.split()
)

In [ ]:
df = df.drop(columns=[c for c in ADMIN_COLUMNS | SUPERSEDED_COLUMNS if c in df.columns])
print(f"Step 1: {len(col_names)} → {df.shape[1]} columns")

### Step 2: turn answer codes into missing values or zeros

BRFSS stores "don't know" and "refused" as numbers. Left as they are, a model reads "refused" (99) as a very large amount.

Which codes a column uses depends on how many digits its answers have. Read it from the column's **largest value in the training rows**:

| largest value | becomes missing (don't know / refused) | becomes 0 ("none") |
|---|---|---|
| 9 | 7, 9 | |
| 99 | 77, 99 | 88 |
| 999 | 777, 999 | 888 |

- Any other largest value: no codes to replace.
- Columns with non-integer values are computed values (BMI, servings per day), not answers: no codes to replace.

**Exceptions**: these columns use their own codes instead of the rule above.

| column | becomes missing | why |
|---|---|---|
| `EMPLOY1` | 9 | 7 = retired and 8 = unable to work are real answers |
| `_RACE` | 9 | 7 = multiracial and 8 = Hispanic are real answers |
| `_AGEG5YR` | 14 | 14 = age unknown, not an age group above 80+ |
| `_AGE65YR` | 3 | age unknown |
| `SEATBELT` | 7, 8, 9 | 8 = never drives or rides in a car, not a point of the always…never scale |
| `MAXVO2_`, `FC60_` | 999 | unknown |
| `DROCDY3_` | 900 | don't know |
| `_DRNKWEK` | 99900 | don't know |


In [ ]:
# "Don't know" / "refused" codes become missing, "none" codes become 0.
# Which codes a column uses is read from its largest value: 9, 99 or 999
CODES_BY_MAX = {
    9: {7: np.nan, 9: np.nan},
    99: {77: np.nan, 99: np.nan, 88: 0},
    999: {777: np.nan, 999: np.nan, 888: 0},
}

# Columns whose codes differ from the rule above (2015 codebook)
SPECIAL_CODES = {
    "EMPLOY1": {9: np.nan},  # 7 = retired and 8 = unable to work are answers
    "_RACE": {9: np.nan},  # 7 = multiracial and 8 = Hispanic are answers
    "_AGEG5YR": {14: np.nan},  # 14 = age unknown, not "older than 80"
    "_AGE65YR": {3: np.nan},
    "SEATBELT": {7: np.nan, 8: np.nan, 9: np.nan},  # 8 = never drives or rides
    "MAXVO2_": {999: np.nan},
    "FC60_": {999: np.nan},
    "DROCDY3_": {900: np.nan},
    "_DRNKWEK": {99900: np.nan},
}


def get_codes(name, column):
    """Codes to replace in one column, from its name or its largest value."""
    if name in SPECIAL_CODES:
        return SPECIAL_CODES[name]
    observed = column.dropna()
    if observed.size == 0 or not np.all(observed == np.round(observed)):
        return {}  # computed values such as BMI or servings per day, not answer codes
    return CODES_BY_MAX.get(observed.max(), {})


df_recoded = df.copy()
n_recoded = 0
for col in df_recoded.columns:
    codes = get_codes(col, df_recoded[col])
    if codes:
        df_recoded[col] = df_recoded[col].replace(codes)
        n_recoded += 1
print(f"Step 2: codes replaced in {n_recoded} of {df_recoded.shape[1]} columns")
print(f"NaN before: {df.isna().sum().sum()} → after: {df_recoded.isna().sum().sum()}")

### Step 3: drop mostly-missing and constant columns

After step 2, drop a column if:
- it is missing in more than 80% of the training rows (many questions belong to optional modules that most states skipped), or
- it has fewer than 2 distinct observed values.

In [ ]:
MAX_MISSING = 0.8  # drop a column missing in more than 80% of training rows

keep_cols = []
for col in df_recoded.columns:
    frac_missing = df_recoded[col].isna().mean()
    n_unique = df_recoded[col].nunique()  # NaN not counted
    if frac_missing <= MAX_MISSING and n_unique >= 2:
        keep_cols.append(col)

df_clean = df_recoded[keep_cols]
print(f"Step 3: {df_recoded.shape[1]} → {df_clean.shape[1]} columns "
      f"({df_recoded.shape[1] - df_clean.shape[1]} dropped)")

### Step 4a: unordered categories → one 0/1 column per category

For these columns the number is a label, not an amount (state 6 is not "more" than state 5):

`_STATE MARITAL EMPLOY1 _RACE BPHIGH4 DIABETE3 RENTHOM1 PERSDOC2 MSCODE`

1. Treat missing as its own category, coded -1 (not answering can be informative).
2. List the categories seen in the training rows.
3. Drop the smallest one: it becomes the baseline, the case where every 0/1 column is 0. Keeping it would make the 0/1 columns add up to the bias column.
4. Make one 0/1 column per remaining category: `values == category`.

On test rows, a missing value or a category never seen in training gives all zeros, like the baseline.

In [ ]:
# Unordered categories: one 0/1 column per category instead of one number
ONE_HOT_COLUMNS = set(
    "_STATE MARITAL EMPLOY1 _RACE BPHIGH4 DIABETE3 RENTHOM1 PERSDOC2 MSCODE".split()
)


one_hot_cols = [col for col in df_clean.columns if col in ONE_HOT_COLUMNS]
oh_features = []
oh_names = []
for col in one_hot_cols:
    values = df_clean[col].values
    values = np.where(np.isnan(values), -1, values)  # missing is its own category
    for cat in np.unique(values)[1:]:  # the smallest category is the baseline
        oh_features.append((values == cat).astype(float))
        oh_names.append(f"{col}={int(cat)}")
df_numeric = df_clean.drop(columns=one_hot_cols)
print(f"Step 4a: {len(one_hot_cols)} categorical columns → {len(oh_features)} 0/1 features")
print(f"Remaining numeric columns: {df_numeric.shape[1]}")

### Step 4b: every other column (amounts and ordered answers)

1. **Fill missing values** with the median of the observed training values (robust to long tails).
2. **Add a 0/1 "was missing" column** if the column has missing values: whether a question was answered is a signal of its own. Questions of the same skipped module are missing in exactly the same rows, so add one flag per distinct missing pattern, not one per column.
3. **Clip outliers** to the 1st and 99th percentiles of the observed training values. Only for columns with more than 10 distinct values: yes/no and short-scale answers have nothing to clip.

In [ ]:
CLIP_PERCENTILES = (1, 99)
MIN_VALUES_TO_CLIP = 10  # yes/no and small ordinal answers are never clipped

num_features = []
num_names = []
seen_missing_patterns = set()
for col in df_numeric.columns:
    values = df_numeric[col].values
    missing = np.isnan(values)
    observed = values[~missing]
    filled = np.where(missing, np.median(observed), values)
    low, high = -np.inf, np.inf
    if np.unique(observed).size > MIN_VALUES_TO_CLIP:
        low, high = np.percentile(observed, CLIP_PERCENTILES)
    num_features.append(np.clip(filled, low, high))
    num_names.append(col)
    # Questions of a skipped module are missing in the same rows: one flag for all of them
    pattern = missing.tobytes()
    if missing.any() and pattern not in seen_missing_patterns:
        num_features.append(missing.astype(float))
        num_names.append(f"{col}_missing")
        seen_missing_patterns.add(pattern)
print(f"Step 4b: {df_numeric.shape[1]} numeric columns → {len(num_features)} features")

### Step 5: standardize

1. Put all the new columns side by side in one float matrix.
2. Compute the mean and the std of each column on the training rows (a std of 0 is replaced by 1).
3. Subtract the mean and divide by the std, on the training rows and the test rows alike, always with the training mean and std.

No bias column here: add the column of 1s when building the model input.

In [ ]:
tx = np.column_stack(oh_features + num_features)  # 0/1 columns first, then the numbers
feature_names = oh_names + num_names
mean, std = tx.mean(axis=0), tx.std(axis=0)
tx -= mean  # in place: tx is about 1 GB on the full training set
tx /= np.where(std > 0, std, 1.0)
print(f"Step 5: {tx.shape[1]} features, {np.isnan(tx).sum()} NaN")

### Reference: the 157 kept columns

Columns that survive steps 1 to 3 on the full training set, in dataset order. **Meaning** is the codebook's title for the column (search the name in `codebook15_llcp.pdf` for the full question and every code). **Processed as** is what step 4a or 4b does with it: *categories* = one 0/1 column per category; *number* = median fill, with a was-missing flag when the column has its own missing pattern, and clipping when it has more than 10 distinct values.

⚠ = unordered labels that step 4b treats as amounts (e.g. place 11 is not "more" than place 2). They should probably be one-hot (step 4a). Most race variants are copies of `_RACE`, which is already one-hot.

| column | meaning | topic | processed as |
|---|---|---|---|
| `_STATE` | State FIPS Code | Record Identification | categories (52 0/1 columns) |
| `NUMADULT` | Number of Adults in Household | Land Line Introduction | number + was-missing flag, clipped |
| `NUMMEN` | Number of Adult men in Household | Land Line Introduction | number + was-missing flag, clipped |
| `NUMWOMEN` | Number of Adult women in Household | Land Line Introduction | number + was-missing flag, clipped |
| `HHADULT` | Number of Adults in Household | Cell Phone Introduction | number + was-missing flag, clipped |
| `GENHLTH` | General Health | Health Status | number + was-missing flag |
| `PHYSHLTH` | Number of Days Physical Health Not Good | Healthy days | number + was-missing flag, clipped |
| `MENTHLTH` | Number of Days Mental Health Not Good | Healthy days | number + was-missing flag, clipped |
| `POORHLTH` | Poor Physical or Mental Health | Healthy days | number + was-missing flag, clipped |
| `HLTHPLN1` | Have any health care coverage | Health Care Access | number + was-missing flag |
| `PERSDOC2` | Has a personal doctor (one / more than one / none) | Health Care Access | categories (3 0/1 columns) |
| `MEDCOST` | Could Not See Doctor Because of Cost | Health Care Access | number + was-missing flag |
| `CHECKUP1` | Length of time since last routine checkup | Health Care Access | number + was-missing flag |
| `BPHIGH4` | Ever Told Blood Pressure High | Hypertension Awareness | categories (4 0/1 columns) |
| `BPMEDS` | Currently Taking Blood Pressure Medication | Hypertension Awareness | number + was-missing flag |
| `BLOODCHO` | Ever Had Blood Cholesterol Checked | Cholesterol Awareness | number + was-missing flag |
| `CHOLCHK` | How Long since Cholesterol Checked | Cholesterol Awareness | number + was-missing flag |
| `TOLDHI2` | Ever Told Blood Cholesterol High | Cholesterol Awareness | number + was-missing flag |
| `CVDSTRK3` | Ever Diagnosed with a Stroke | Chronic Health Conditions | number + was-missing flag |
| `ASTHMA3` | Ever Told Had Asthma | Chronic Health Conditions | number + was-missing flag |
| `CHCSCNCR` | Ever told you had skin cancer? | Chronic Health Conditions | number + was-missing flag |
| `CHCOCNCR` | Ever told you had any other types of cancer? | Chronic Health Conditions | number + was-missing flag |
| `CHCCOPD1` | Ever told you have chronic obstructive pulmonary disease, emphysema or chronic bronchitis? | Chronic Health Conditions | number + was-missing flag |
| `HAVARTH3` | Told Have Arthritis | Chronic Health Conditions | number + was-missing flag |
| `ADDEPEV2` | Ever told you had a depressive disorder | Chronic Health Conditions | number + was-missing flag |
| `CHCKIDNY` | Ever told you have kidney disease? | Chronic Health Conditions | number + was-missing flag |
| `DIABETE3` | Ever told you have diabetes | Chronic Health Conditions | categories (4 0/1 columns) |
| `SEX` | Respondents Sex | Demographics | number |
| `MARITAL` | Marital Status | Demographics | categories (6 0/1 columns) |
| `EDUCA` | Education Level | Demographics | number + was-missing flag |
| `RENTHOM1` | Own or Rent Home | Demographics | categories (3 0/1 columns) |
| `VETERAN3` | Are You A Veteran | Demographics | number + was-missing flag |
| `EMPLOY1` | Employment Status | Demographics | categories (8 0/1 columns) |
| `CHILDREN` | Number of Children in Household | Demographics | number + was-missing flag, clipped |
| `INCOME2` | Income Level | Demographics | number + was-missing flag |
| `INTERNET` | Internet use in the past 30 days? | Demographics | number + was-missing flag |
| `QLACTLM2` | Activity Limitation Due to Health Problems | Demographics | number + was-missing flag |
| `USEEQUIP` | Health Problems Requiring Special Equipment | Demographics | number + was-missing flag |
| `BLIND` | Blind or Difficulty seeing | Demographics | number + was-missing flag |
| `DECIDE` | Difficulty Concentrating or Remembering | Demographics | number + was-missing flag |
| `DIFFWALK` | Difficulty Walking or Climbing Stairs | Demographics | number + was-missing flag |
| `DIFFDRES` | Difficulty Dressing or Bathing | Demographics | number + was-missing flag |
| `DIFFALON` | Difficulty Doing Errands Alone | Demographics | number + was-missing flag |
| `SMOKE100` | Smoked at Least 100 Cigarettes | Tobacco Use | number + was-missing flag |
| `SMOKDAY2` | Frequency of Days Now Smoking | Tobacco Use | number + was-missing flag |
| `LASTSMK2` | Interval Since Last Smoked | Tobacco Use | number + was-missing flag |
| `USENOW3` | Use of Smokeless Tobacco Products | Tobacco Use | number + was-missing flag |
| `AVEDRNK2` | Avg alcoholic drinks per day in past 30 | Alcohol Consumption | number + was-missing flag, clipped |
| `DRNK3GE5` | Binge Drinking | Alcohol Consumption | number + was-missing flag, clipped |
| `MAXDRNKS` | Most drinks on single occasion past 30 days | Alcohol Consumption | number + was-missing flag, clipped |
| `EXERANY2` | Exercise in Past 30 Days | Exercise (Physical Activity) | number + was-missing flag |
| `LMTJOIN3` | Limited Because of Joint Symptoms | Arthritis Burden | number + was-missing flag |
| `ARTHDIS2` | Does Arthritis Affect Whether You Work | Arthritis Burden | number + was-missing flag |
| `ARTHSOCL` | Social Activities Limited Because of Joint Symptoms | Arthritis Burden | number + was-missing flag |
| `JOINPAIN` | How Bad Was Joint Pain | Arthritis Burden | number + was-missing flag, clipped |
| `SEATBELT` | How often use seatbelts in car? | Seatbelt Use | number + was-missing flag |
| `FLUSHOT6` | Adult flu shot/spray past 12 mos | Immunization | number + was-missing flag |
| `IMFVPLAC` | Where did you get your last flu shot/vaccine? | Immunization | ⚠ number + was-missing flag, clipped (1-11 = places (doctor, store, workplace, ...)) |
| `PNEUVAC3` | Pneumonia shot ever | Immunization | number + was-missing flag |
| `HIVTST6` | Ever tested HIV | HIV/AIDS | number + was-missing flag |
| `WHRTST10` | Location of last HIV test | HIV/AIDS | ⚠ number + was-missing flag (1-9 = places (doctor, clinic, prison, home, ...)) |
| `CAREGIV1` | Provided regular care for family or friend | Caregiver | ⚠ number + was-missing flag (1 = yes, 2 = no, 8 = care recipient died) |
| `CIMEMLOS` | Have you experienced confusion or memory loss that is happening more often or is getting worse? | Cognitive Decline | number + was-missing flag |
| `SXORIENT` | Sexual orientation or gender identity | Sexual Orientation and Gender Identity | ⚠ number + was-missing flag (1-4 = straight, gay, bisexual, other) |
| `TRNSGNDR` | Do you consider yourself to be transgender? | Sexual Orientation and Gender Identity | ⚠ number + was-missing flag (1-3 = types of transgender, 4 = no) |
| `MSCODE` | Metropolitan Status Code | Weighting Variables | categories (4 0/1 columns) |
| `_RFHLTH` | Adults with good or better health | Calculated Variables | number |
| `_HCVU651` | Respondents aged 18-64 with health care coverage | Calculated Variables | number + was-missing flag |
| `_RFHYPE5` | High Blood Pressure Calculated Variable | Calculated Variables | number + was-missing flag |
| `_CHOLCHK` | Cholesterol Checked Calculated Variable | Calculated Variables | number + was-missing flag |
| `_RFCHOL` | High Cholesterol Calculated Variable | Calculated Variables | number |
| `_LTASTH1` | Lifetime Asthma Calculated Variable | Calculated Variables | number |
| `_CASTHM1` | Current Asthma Calculated Variable | Calculated Variables | number + was-missing flag |
| `_ASTHMS1` | Computed Asthma Status | Calculated Variables | number |
| `_DRDXAR1` | Respondents diagnosed with arthritis | Calculated Variables | number |
| `_PRACE1` | Computed Preferred Race | Calculated Race Variables | ⚠ number + was-missing flag (1-8 = races) |
| `_MRACE1` | Calculated non-Hispanic Race including multiracial | Calculated Race Variables | ⚠ number (1-7 = races) |
| `_HISPANC` | Hispanic, Latino/a, or Spanish origin calculated variable | Calculated Race Variables | number + was-missing flag |
| `_RACE` | Computed Race-Ethnicity grouping | Calculated Race Variables | categories (8 0/1 columns) |
| `_RACEG21` | Non-Hispanic white vs all others | Calculated Race Variables | number + was-missing flag |
| `_RACEGR3` | Computed Five level race/ethnicity category | Calculated Race Variables | ⚠ number (1-5 = race/ethnicity groups) |
| `_RACE_G1` | Computed race groups used for internet prevalence tables | Calculated Race Variables | ⚠ number (1-5 = race/ethnicity groups) |
| `_AGEG5YR` | Reported age in five-year age categories calculated variable | Calculated Variables | number + was-missing flag, clipped |
| `_AGE65YR` | Reported age in two age groups calculated variable | Calculated Variables | number |
| `_AGE80` | Imputed Age value collapsed above 80 | Calculated Variables | number, clipped |
| `_AGE_G` | Imputed age in six groups | Calculated Variables | number |
| `HTIN4` | Computed Height in Inches | Calculated Variables | number + was-missing flag, clipped |
| `HTM4` | Computed Height in Meters | Calculated Variables | number + was-missing flag, clipped |
| `WTKG3` | Computed Weight in Kilograms | Calculated Variables | number + was-missing flag, clipped |
| `_BMI5` | Computed body mass index | Calculated Variables | number + was-missing flag, clipped |
| `_BMI5CAT` | Computed body mass index categories | Calculated Variables | number |
| `_RFBMI5` | Overweight or obese calculated variable | Calculated Variables | number |
| `_CHLDCNT` | Computed number of children in household | Calculated Variables | number + was-missing flag |
| `_EDUCAG` | Computed level of education completed categories | Calculated Variables | number |
| `_INCOMG` | Computed income categories | Calculated Variables | number |
| `_SMOKER3` | Computed Smoking Status | Calculated Variables | number + was-missing flag |
| `_RFSMOK3` | Current Smoking Calculated Variable | Calculated Variables | number |
| `DRNKANY5` | Drink any alcoholic beverages in past 30 days | Calculated Variables | number + was-missing flag |
| `DROCDY3_` | Computed drink-occasions-per-day | Calculated Variables | number, clipped |
| `_RFBING5` | Binge Drinking Calculated Variable | Calculated Variables | number + was-missing flag |
| `_DRNKWEK` | Computed number of drinks of alcohol beverages per week | Calculated Variables | number + was-missing flag, clipped |
| `_RFDRHV5` | Heavy Alcohol Consumption Calculated Variable | Calculated Variables | number |
| `FTJUDA1_` | Computed Fruit Juice intake in times per day | Calculated Variables | number + was-missing flag, clipped |
| `FRUTDA1_` | Computed Fruit intake in times per day | Calculated Variables | number + was-missing flag, clipped |
| `BEANDAY_` | Computed Bean intake in times per day | Calculated Variables | number + was-missing flag, clipped |
| `GRENDAY_` | Computed Dark Green Vegetable intake in times per day | Calculated Variables | number + was-missing flag, clipped |
| `ORNGDAY_` | Computed Orange-Colored Vegetable intake in times per day | Calculated Variables | number + was-missing flag, clipped |
| `VEGEDA1_` | Computed Other Vegetable intake in times per day | Calculated Variables | number + was-missing flag, clipped |
| `_MISFRTN` | The number of missing fruit responses | Calculated Variables | number |
| `_MISVEGN` | The number of missing vegetable responses | Calculated Variables | number |
| `_FRTRESP` | Missing any fruit responses | Calculated Variables | number |
| `_VEGRESP` | Missing any vegetable responses | Calculated Variables | number |
| `_FRUTSUM` | Total fruits consumed per day | Calculated Variables | number + was-missing flag, clipped |
| `_VEGESUM` | Total vegetables consumed per day | Calculated Variables | number + was-missing flag, clipped |
| `_FRTLT1` | Consume Fruit 1 or more times per day | Calculated Variables | number |
| `_VEGLT1` | Consume Vegetables 1 or more times per day | Calculated Variables | number |
| `_FRT16` | Reported consuming Fruit >16 per day | Calculated Variables | number |
| `_VEG23` | Reported consuming Vegetables >23 per day | Calculated Variables | number |
| `_FRUITEX` | Fruit Exclusion from analyses | Calculated Variables | number |
| `_VEGETEX` | Vegetable Exclusion from analyses | Calculated Variables | number |
| `_TOTINDA` | Leisure Time Physical Activity Calculated Variable | Calculated Variables | number |
| `METVL11_` | Activity MET Value for First Activity | Calculated Variables | number + was-missing flag, clipped |
| `METVL21_` | Activity MET Value for Second Activity | Calculated Variables | number + was-missing flag, clipped |
| `MAXVO2_` | Estimated Age-Gender Specific Maximum Oxygen Consumption | Calculated Variables | number, clipped |
| `FC60_` | Estimated Functional Capacity | Calculated Variables | number, clipped |
| `ACTIN11_` | Estimated Activity Intensity for First Activity | Calculated Variables | number + was-missing flag |
| `ACTIN21_` | Estimated Activity Intensity for Second Activity | Calculated Variables | number + was-missing flag |
| `PADUR1_` | Minutes of First Activity | Calculated Variables | number + was-missing flag, clipped |
| `PADUR2_` | Minutes of Second Activity | Calculated Variables | number + was-missing flag, clipped |
| `PAFREQ1_` | Physical Activity Frequency per Week for First Activity | Calculated Variables | number + was-missing flag, clipped |
| `PAFREQ2_` | Physical Activity Frequency per Week for Second Activity | Calculated Variables | number + was-missing flag, clipped |
| `_MINAC11` | Minutes of Physical Activity per week for First Activity | Calculated Variables | number + was-missing flag, clipped |
| `_MINAC21` | Minutes of Physical Activity per week for Second Activity | Calculated Variables | number + was-missing flag, clipped |
| `STRFREQ_` | Strength Activity Frequency per Week | Calculated Variables | number + was-missing flag, clipped |
| `PAMISS1_` | Missing Physical Activity Data | Calculated Variables | number |
| `PAMIN11_` | Minutes of Physical Activity per week for First Activity | Calculated Variables | number + was-missing flag, clipped |
| `PAMIN21_` | Minutes of Physical Activity per week for Second Activity | Calculated Variables | number + was-missing flag, clipped |
| `PA1MIN_` | Minutes of total Physical Activity per week | Calculated Variables | number + was-missing flag, clipped |
| `PAVIG11_` | Minutes of Vigorous Physical Activity per week for First Activity | Calculated Variables | number + was-missing flag, clipped |
| `PAVIG21_` | Minutes of Vigorous Physical Activity per week for Second Activity | Calculated Variables | number + was-missing flag, clipped |
| `PA1VIGM_` | Minutes of total Vigorous Physical Activity per week | Calculated Variables | number + was-missing flag, clipped |
| `_PACAT1` | Physical Activity Categories | Calculated Variables | number + was-missing flag |
| `_PAINDX1` | Physical Activity Index | Calculated Variables | number + was-missing flag |
| `_PA150R2` | 150 Minute Physical Activity Calculated Variable | Calculated Variables | number |
| `_PA300R2` | 300 Minute Physical Activity Calculated Variable | Calculated Variables | number |
| `_PA30021` | 300 Minute Physical Activity 2-Level Calculated Variable | Calculated Variables | number |
| `_PASTRNG` | Muscle Strengthening Recommendation | Calculated Variables | number |
| `_PAREC1` | Aerobic and Strengthening Guideline | Calculated Variables | number + was-missing flag |
| `_PASTAE1` | Aerobic and Strengthening (2-level) | Calculated Variables | number |
| `_LMTACT1` | Limited usual activities | Calculated Variables | number + was-missing flag |
| `_LMTWRK1` | Limited work activities | Calculated Variables | number + was-missing flag |
| `_LMTSCL1` | Limited social activities | Calculated Variables | number + was-missing flag |
| `_RFSEAT2` | Always or Nearly Always Wear Seat Belts | Calculated Variables | number + was-missing flag |
| `_RFSEAT3` | Always Wear Seat Belts | Calculated Variables | number |
| `_FLSHOT6` | Flu Shot Calculated Variable | Calculated Variables | number + was-missing flag |
| `_PNEUMO2` | Pneumonia Vaccination Calculated Variable | Calculated Variables | number + was-missing flag |
| `_AIDTST3` | Ever been tested for HIV calculated variable | Calculated Variables | number |